# Classic Component Making
- create non-modeled components

In [ ]:
RUN_TYPE = "prod"
GIT_CHECKOUT = "feature/B-3046295"
MODEL_VERSION = "2.0.0"
PKG_VERSION = "10.0.0"
HELPERS_VERSION = "2.0.0"
LAKEHOUSE = "CLV_lakehouse_classic_spl_STG"

## Setup

In [ ]:
import sys

import notebookutils

SPL_ROOT = f"{notebookutils.nbResPath}/env/classic_specialty_ltv-{PKG_VERSION}"
HELPER_ROOT = f"{notebookutils.nbResPath}/env/ltv_helpers-{HELPERS_VERSION}"

for root in (SPL_ROOT, HELPER_ROOT):
    if root not in sys.path:
        sys.path.insert(0, root)

import classic_spl_ltv
import ltv_helpers

assert notebookutils.runtime.context["defaultLakehouseName"] == LAKEHOUSE, f"Attach {LAKEHOUSE}"
assert classic_spl_ltv.__file__.startswith(SPL_ROOT), "Restart the session"
assert ltv_helpers.__file__.startswith(HELPER_ROOT), "Restart the session"

In [ ]:
from classic_spl_ltv.config import run_paths

p = run_paths.make_paths(RUN_TYPE, GIT_CHECKOUT, MODEL_VERSION)
BASE = f"Files/data/classic_spl_ltv/{RUN_TYPE}/{GIT_CHECKOUT}"

## Non-modeled components

In [ ]:
import classic_spl_ltv.jobs.balance.process_bal_targets as bal
import classic_spl_ltv.jobs.expense.process_raw_expense as prexp
import classic_spl_ltv.jobs.finance.cat_loads as cat
import classic_spl_ltv.jobs.risk.make_constants as coc
import classic_spl_ltv.jobs.risk.ncor as ncor

### Expense

In [ ]:
prexp.main(spark)

### Cat Loss

In [ ]:
cat.main(spark)

### Cost of Capital

In [ ]:
coc.main(spark)

### Reinsurance Ratios

In [ ]:
ncor.main(spark)

### Balance Target

In [ ]:
bal.main(spark)

## Outputs

In [ ]:
from delta.tables import DeltaTable


def delta_tables(root):
    dirs = [f for f in notebookutils.fs.ls(root) if f.isDir]
    if any(f.name == "_delta_log" for f in dirs):
        return [root]
    return [t for f in dirs if f.name != "eltv_pipeline" for t in delta_tables(f.path)]


for path in delta_tables(BASE):
    last = DeltaTable.forPath(spark, path).history(1).first()
    rows = spark.read.format("delta").load(path).count()
    print(f"{last['timestamp']}  {rows:>12,}  {path.split(GIT_CHECKOUT)[-1]}")